In [ ]:
from google.colab import files

uploaded = files.upload()


Saving Agri-LLaVA_Knowledge-Infused_Large_Multimodal_Assi.pdf to Agri-LLaVA_Knowledge-Infused_Large_Multimodal_Assi.pdf


In [ ]:
import os
os.listdir()


['.config',
 'mathematics-13-00566-v2.pdf',
 'Agri-LLaVA_Knowledge-Infused_Large_Multimodal_Assi.pdf',
 'sample_data']

Check if the PDF is digital or scanned

In [ ]:
!pip install pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 63.9 MB/s eta 0:00:00


Function to detect digital vs scanned PDF

In [ ]:
import pdfplumber

def is_digital_pdf(pdf_path, max_pages=3):
    """
    Checks first few pages for extractable text.
    Returns True if digital, False if scanned.
    """
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages[:max_pages]):
            text = page.extract_text()
            if text and text.strip():
                return True
    return False


In [ ]:
pdf_files = [
    "mathematics-13-00566-v2.pdf",
    "Agri-LLaVA_Knowledge-Infused_Large_Multimodal_Assi.pdf"
]

for pdf in pdf_files:
    print(pdf, "→ Digital PDF?" , is_digital_pdf(pdf))

mathematics-13-00566-v2.pdf → Digital PDF? True
Agri-LLaVA_Knowledge-Infused_Large_Multimodal_Assi.pdf → Digital PDF? True


In [ ]:
!apt-get update
!apt-get install -y ghostscript python3-tk


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,302 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,619 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:13 http://security.ubuntu.com/ubuntu j

In [ ]:
!pip install camelot-py[cv]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.7 MB/s eta 0:00:00


In [ ]:
import camelot
camelot.__version__


'1.0.9'

In [ ]:
pdf_path = "mathematics-13-00566-v2.pdf"

tables = camelot.read_pdf(
    pdf_path,
    flavor="stream",
    pages="all"
)

print("Total tables found:", tables.n)


/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (156.244, 493.03003999999993, 573.9351360000002, 848.4965983943661)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


Total tables found: 40


table filter

In [ ]:
import re
import numpy as np

def is_real_table(df, min_rows=3, min_cols=2):
    # basic shape check
    if df.shape[0] < min_rows or df.shape[1] < min_cols:
        return False

    # remove empty rows
    df = df.replace("", np.nan).dropna(how="all")

    # average cell length (remove paragraphs)
    avg_cell_length = (
        df.astype(str)
          .stack()
          .str.len()
          .mean()
    )
    if avg_cell_length > 40:
        return False

    # count numeric cells
    numeric_mask = df.astype(str).apply(
        lambda col: col.str.contains(r"\d", regex=True)
    )
    numeric_cells = numeric_mask.sum().sum()
    total_cells = df.size

    numeric_ratio = numeric_cells / total_cells

    # require enough numeric content
    if numeric_ratio < 0.15:
        return False

    # require at least one column with repeated numbers
    column_numeric_counts = numeric_mask.sum()
    if column_numeric_counts.max() < 3:
        return False

    return True

Filter Camelot tables

In [ ]:
real_tables = []
junk_tables = []

for i, table in enumerate(tables):
    df = table.df
    if is_real_table(df):
        real_tables.append((i, df))
    else:
        junk_tables.append((i, df))

print("Real tables:", len(real_tables))
print("Junk tables:", len(junk_tables))


Real tables: 10
Junk tables: 30


Look at real table indices

In [ ]:
[i for i, _ in real_tables[:5]]


[1, 7, 8, 11, 18]

In [ ]:
[i for i, _ in real_tables]


[1, 7, 8, 11, 18, 23, 24, 26, 27, 28]

In [ ]:
table_id, df = real_tables[0]
print("Table ID:", table_id)
df


Table ID: 1


,0,1,2
0,", Qijie Fan 1,2, Xuan Chen 1,2, Min Li 2, Zeyi...",,
1,1,,
2,,"School of Software, Shanxi Agricultural Univer...",
3,,,s20222093@stu.sxau.edu.cn (Y.Z.); s20222095@st...
4,2,,
5,,,Agricultural Information Institute of Chinese ...
6,,limin@caas.cn,
7,3,,
8,,Guizhou Agricultural Science and Technology In...,
9,,zeyingzhao@126.com,
